# Libraries

In [2]:
from common import *
engine = get_engine()

# Testing if bronze working

In [2]:
count = pd.read_sql(text(f"SELECT COUNT(*) AS row_count  FROM bronze.flights"), engine)
count

,row_count
0,5812365


In [3]:
count = pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.weather"), engine)
count

,row_count
0,2225040


In [4]:
pd.read_sql(text(f"SELECT COUNT(*) AS row_count FROM bronze.airlines"), engine)

,row_count
0,14


In [5]:
pd.read_sql(text(f'SELECT COUNT(*) AS row_count FROM bronze.airports'), engine)

,row_count
0,322


# Joining the datasets

## Column names

In [6]:
for table in ["airlines", "airports", "flights", "weather"]:
    columns = pd.read_sql(
        text(f"""
            SELECT COLUMN_NAME
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = 'bronze'
              AND TABLE_NAME = '{table}'
            ORDER BY ORDINAL_POSITION
        """),
        engine
    )["COLUMN_NAME"].tolist()

    print(f"\n{table}:")
    print(columns)


airlines:
['IATA_CODE', 'AIRLINE']

airports:
['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']

flights:
['MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'DISTANCE', 'SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL', 'DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'CANCELLED', 'DIVERTED', 'TAXI_OUT', 'TAXI_IN', 'AIR_TIME']

weather:
['airport_code', 'observation_time', 'Latitude', 'Longitude', 'temperature', 'precipitation', 'wind_speed', 'visibility', 'weather_condition', 'surface_pressure', 'wind_direction', 'snowfall', 'wind_gusts']


## join

### join of 3 datatset

In [4]:
query = text("""
    SELECT
        f.*,

        -- Airline məlumatları
        al.AIRLINE AS AIRLINE_NAME,

        -- Başlanğıc hava limanı
        origin.AIRPORT AS ORIGIN_AIRPORT_NAME,
        origin.CITY AS ORIGIN_CITY,
        origin.STATE AS ORIGIN_STATE,
        origin.COUNTRY AS ORIGIN_COUNTRY,

        -- Təyinat hava limanı
        destination.AIRPORT AS DESTINATION_AIRPORT_NAME,
        destination.CITY AS DESTINATION_CITY,
        destination.STATE AS DESTINATION_STATE,
        destination.COUNTRY AS DESTINATION_COUNTRY

    FROM bronze.flights AS f
    LEFT JOIN bronze.airlines AS al
        ON f.AIRLINE = al.IATA_CODE
    LEFT JOIN bronze.airports AS origin
        ON f.ORIGIN_AIRPORT = origin.IATA_CODE

    LEFT JOIN bronze.airports AS destination
        ON f.DESTINATION_AIRPORT = destination.IATA_CODE
""")

In [5]:
chunks = []
total_rows = 0

for chunk in pd.read_sql(query, engine, chunksize=100_000):
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f"Yüklənib: {total_rows:,} sətir")

df_flights = pd.concat(chunks, ignore_index=True)

print("\nBirləşdirmə tamamlandı.")
print("Dataset ölçüsü:", df_flights.shape)

Yüklənib: 100,000 sətir
Yüklənib: 200,000 sətir
Yüklənib: 300,000 sətir
Yüklənib: 400,000 sətir
Yüklənib: 500,000 sətir
Yüklənib: 600,000 sətir
Yüklənib: 700,000 sətir
Yüklənib: 800,000 sətir
Yüklənib: 900,000 sətir
Yüklənib: 1,000,000 sətir
Yüklənib: 1,100,000 sətir
Yüklənib: 1,200,000 sətir
Yüklənib: 1,300,000 sətir
Yüklənib: 1,400,000 sətir
Yüklənib: 1,500,000 sətir
Yüklənib: 1,600,000 sətir
Yüklənib: 1,700,000 sətir
Yüklənib: 1,800,000 sətir
Yüklənib: 1,900,000 sətir
Yüklənib: 2,000,000 sətir
Yüklənib: 2,100,000 sətir
Yüklənib: 2,200,000 sətir
Yüklənib: 2,300,000 sətir
Yüklənib: 2,400,000 sətir
Yüklənib: 2,500,000 sətir
Yüklənib: 2,600,000 sətir
Yüklənib: 2,700,000 sətir
Yüklənib: 2,800,000 sətir
Yüklənib: 2,900,000 sətir
Yüklənib: 3,000,000 sətir
Yüklənib: 3,100,000 sətir
Yüklənib: 3,200,000 sətir
Yüklənib: 3,300,000 sətir
Yüklənib: 3,400,000 sətir
Yüklənib: 3,500,000 sətir
Yüklənib: 3,600,000 sətir
Yüklənib: 3,700,000 sətir
Yüklənib: 3,800,000 sətir
Yüklənib: 3,900,000 sətir
Yükl

In [6]:
query_missing_summary = text("""
    SELECT
        COUNT(*) AS TOTAL_FLIGHTS,

        SUM(CASE
            WHEN al.IATA_CODE IS NULL THEN 1
            ELSE 0
        END) AS MISSING_AIRLINE,

        SUM(CASE
            WHEN origin.IATA_CODE IS NULL THEN 1
            ELSE 0
        END) AS MISSING_ORIGIN_AIRPORT,

        SUM(CASE
            WHEN destination.IATA_CODE IS NULL THEN 1
            ELSE 0
        END) AS MISSING_DESTINATION_AIRPORT

    FROM bronze.flights AS f

    LEFT JOIN bronze.airlines AS al
        ON f.AIRLINE = al.IATA_CODE

    LEFT JOIN bronze.airports AS origin
        ON f.ORIGIN_AIRPORT = origin.IATA_CODE

    LEFT JOIN bronze.airports AS destination
        ON f.DESTINATION_AIRPORT = destination.IATA_CODE
""")

df_missing_summary = pd.read_sql(query_missing_summary, engine)

df_missing_summary

,TOTAL_FLIGHTS,MISSING_AIRLINE,MISSING_ORIGIN_AIRPORT,MISSING_DESTINATION_AIRPORT
0,5812365,0,1011,1010


In [6]:
import gc

chunk_size = 100_000
cleaned_chunks = []

rows_before = len(df_flights)
rows_removed = 0

for start in range(0, rows_before, chunk_size):
    end = min(start + chunk_size, rows_before)

    # Cari hissəni götür
    chunk = df_flights.iloc[start:end]

    # Hava limanı məlumatları olmayan sətirləri sil
    clean_chunk = chunk.dropna(
        subset=[
            "ORIGIN_AIRPORT_NAME",
            "DESTINATION_AIRPORT_NAME"
        ]
    )

    rows_removed += len(chunk) - len(clean_chunk)
    cleaned_chunks.append(clean_chunk)

    print(
        f"Emal edildi: {end:,}/{rows_before:,} | "
        f"Silinib: {rows_removed:,}"
    )

# Təmizlənmiş hissələri birləşdir
df_flights_clean = pd.concat(
    cleaned_chunks,
    ignore_index=True
)

print("\nTəmizləmə tamamlandı.")
print(f"Əvvəlki sətir sayı: {rows_before:,}")
print(f"Silinən sətir sayı: {rows_removed:,}")
print(f"Qalan sətir sayı: {len(df_flights_clean):,}")

del cleaned_chunks
gc.collect()

Emal edildi: 100,000/5,812,365 | Silinib: 0
Emal edildi: 200,000/5,812,365 | Silinib: 0
Emal edildi: 300,000/5,812,365 | Silinib: 0
Emal edildi: 400,000/5,812,365 | Silinib: 0
Emal edildi: 500,000/5,812,365 | Silinib: 0
Emal edildi: 600,000/5,812,365 | Silinib: 0
Emal edildi: 700,000/5,812,365 | Silinib: 0
Emal edildi: 800,000/5,812,365 | Silinib: 0
Emal edildi: 900,000/5,812,365 | Silinib: 0
Emal edildi: 1,000,000/5,812,365 | Silinib: 0
Emal edildi: 1,100,000/5,812,365 | Silinib: 0
Emal edildi: 1,200,000/5,812,365 | Silinib: 0
Emal edildi: 1,300,000/5,812,365 | Silinib: 0
Emal edildi: 1,400,000/5,812,365 | Silinib: 0
Emal edildi: 1,500,000/5,812,365 | Silinib: 0
Emal edildi: 1,600,000/5,812,365 | Silinib: 0
Emal edildi: 1,700,000/5,812,365 | Silinib: 0
Emal edildi: 1,800,000/5,812,365 | Silinib: 0
Emal edildi: 1,900,000/5,812,365 | Silinib: 0
Emal edildi: 2,000,000/5,812,365 | Silinib: 0
Emal edildi: 2,100,000/5,812,365 | Silinib: 0
Emal edildi: 2,200,000/5,812,365 | Silinib: 0
Emal e

Emal edildi: 4,800,000/5,812,365 | Silinib: 1,814
Emal edildi: 4,900,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,000,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,100,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,200,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,300,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,400,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,500,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,600,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,700,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,800,000/5,812,365 | Silinib: 2,021
Emal edildi: 5,812,365/5,812,365 | Silinib: 2,021

Təmizləmə tamamlandı.
Əvvəlki sətir sayı: 5,812,365
Silinən sətir sayı: 2,021
Qalan sətir sayı: 5,810,344


0

In [8]:
df_missing_summary_clean = pd.DataFrame([{
    "TOTAL_FLIGHTS": len(df_flights_clean),

    "MISSING_AIRLINE": (
        df_flights_clean["AIRLINE_NAME"].isna().sum()
    ),

    "MISSING_ORIGIN_AIRPORT": (
        df_flights_clean["ORIGIN_AIRPORT_NAME"].isna().sum()
    ),

    "MISSING_DESTINATION_AIRPORT": (
        df_flights_clean["DESTINATION_AIRPORT_NAME"].isna().sum()
    )
}])

df_missing_summary_clean

,TOTAL_FLIGHTS,MISSING_AIRLINE,MISSING_ORIGIN_AIRPORT,MISSING_DESTINATION_AIRPORT
0,5810344,0,0,0


In [8]:
query_weather_duplicates = text("""
    SELECT TOP 20
        airport_code,
        CAST(observation_time AS DATE) AS WEATHER_DATE,
        COUNT(*) AS OBSERVATION_COUNT
    FROM bronze.weather
    GROUP BY
        airport_code,
        CAST(observation_time AS DATE)
    HAVING COUNT(*) > 1
    ORDER BY OBSERVATION_COUNT DESC
""")

df_weather_duplicates = pd.read_sql(
    query_weather_duplicates,
    engine
)

df_weather_duplicates

,airport_code,WEATHER_DATE,OBSERVATION_COUNT
0,ABE,2015-01-01,24
1,ABI,2015-01-01,24
2,ABQ,2015-01-01,24
3,ABR,2015-01-01,24
4,ABY,2015-01-01,24
5,ACK,2015-01-01,24
6,ACT,2015-01-01,24
7,ACV,2015-01-01,24
8,ACY,2015-01-01,24
9,ADK,2015-01-01,24


In [39]:
query_weather_time_check = text("""
    SELECT TOP 30
        airport_code,
        observation_time,
        temperature,
        precipitation,
        wind_speed
    FROM bronze.weather
    WHERE airport_code = 'ABE'
      AND CAST(observation_time AS DATE) = '2015-01-01'
    ORDER BY observation_time
""")

df_weather_time_check = pd.read_sql(
    query_weather_time_check,
    engine
)

display(df_weather_time_check)

print(df_weather_time_check["observation_time"].tolist())

,airport_code,observation_time,temperature,precipitation,wind_speed
0,ABE,2015-01-01 00:00:00,-4.4,0.0,7.7
1,ABE,2015-01-01 01:00:00,-4.7,0.0,8.0
2,ABE,2015-01-01 02:00:00,-4.9,0.0,10.4
3,ABE,2015-01-01 03:00:00,-5.0,0.0,12.1
4,ABE,2015-01-01 04:00:00,-5.0,0.0,12.0
5,ABE,2015-01-01 05:00:00,-4.9,0.0,12.7
6,ABE,2015-01-01 06:00:00,-4.7,0.0,14.5
7,ABE,2015-01-01 07:00:00,-4.9,0.0,15.0
8,ABE,2015-01-01 08:00:00,-4.9,0.0,15.3
9,ABE,2015-01-01 09:00:00,-5.0,0.0,15.0


[Timestamp('2015-01-01 00:00:00'), Timestamp('2015-01-01 01:00:00'), Timestamp('2015-01-01 02:00:00'), Timestamp('2015-01-01 03:00:00'), Timestamp('2015-01-01 04:00:00'), Timestamp('2015-01-01 05:00:00'), Timestamp('2015-01-01 06:00:00'), Timestamp('2015-01-01 07:00:00'), Timestamp('2015-01-01 08:00:00'), Timestamp('2015-01-01 09:00:00'), Timestamp('2015-01-01 10:00:00'), Timestamp('2015-01-01 11:00:00'), Timestamp('2015-01-01 12:00:00'), Timestamp('2015-01-01 13:00:00'), Timestamp('2015-01-01 14:00:00'), Timestamp('2015-01-01 15:00:00'), Timestamp('2015-01-01 16:00:00'), Timestamp('2015-01-01 17:00:00'), Timestamp('2015-01-01 18:00:00'), Timestamp('2015-01-01 19:00:00'), Timestamp('2015-01-01 20:00:00'), Timestamp('2015-01-01 21:00:00'), Timestamp('2015-01-01 22:00:00'), Timestamp('2015-01-01 23:00:00')]


In [13]:

# 1. Prepare the cleaned flights data

print("DataFrame mövcuddur:", "df_flights_clean" in globals())

if "df_flights_clean" in globals():
    print("Sətir sayı:", len(df_flights_clean))
    print("Sütun sayı:", len(df_flights_clean.columns))
    print("Yaddaş (MB):", round(
        df_flights_clean.memory_usage(deep=True).sum() / 1024**2, 2
    ))
    print("Lazım olan sütunlar:", [
        col for col in ["MONTH", "DAY", "SCHEDULED_DEPARTURE",
                        "ORIGIN_AIRPORT"]
        if col in df_flights_clean.columns
    ])



DataFrame mövcuddur: True
Sətir sayı: 5810344
Sütun sayı: 30
Yaddaş (MB): 2056.77
Lazım olan sütunlar: ['MONTH', 'DAY', 'SCHEDULED_DEPARTURE', 'ORIGIN_AIRPORT']


In [41]:
weather_stats = pd.read_sql(
    text("""
        SELECT
            COUNT_BIG(*) AS total_rows,
            COUNT(DISTINCT airport_code) AS airport_count,
            MIN(observation_time) AS first_observation,
            MAX(observation_time) AS last_observation
        FROM bronze.weather
    """),
    engine
)

display(weather_stats)


,total_rows,airport_count,first_observation,last_observation
0,2225040,254,2015-01-01,2015-12-31 23:00:00


In [9]:
# Settings
chunk_size = 80000
target_table = "flights_total"
total_written = 0

# Load weather data from SQL Server
weather_query = text("""
    SELECT
        airport_code,
        observation_time,
        temperature AS WEATHER_TEMPERATURE,
        precipitation AS WEATHER_PRECIPITATION,
        wind_speed AS WEATHER_WIND_SPEED,
        visibility AS WEATHER_VISIBILITY,
        weather_condition AS WEATHER_CONDITION,
        surface_pressure AS WEATHER_SURFACE_PRESSURE,
        wind_direction AS WEATHER_WIND_DIRECTION,
        snowfall AS WEATHER_SNOWFALL,
        wind_gusts AS WEATHER_WIND_GUSTS
    FROM bronze.weather
""")

df_weather = pd.read_sql(weather_query, engine)

df_weather["observation_time"] = pd.to_datetime(
    df_weather["observation_time"]
)


In [47]:

# Prepare weather lookup using airport and observation hour
df_weather = df_weather.drop_duplicates(
    subset=["airport_code", "observation_time"]
)

# Process cleaned flights without copying the entire DataFrame
for start in range(0, len(df_flights_clean), chunk_size):
    end = min(start + chunk_size, len(df_flights_clean))

    # Copy only the current chunk
    chunk = df_flights_clean.iloc[start:end].copy()

    # Build scheduled flight datetime
    flight_date = pd.to_datetime({
        "year": 2015,
        "month": chunk["MONTH"],
        "day": chunk["DAY"]
    }, errors="coerce")

    departure = pd.to_numeric(
        chunk["SCHEDULED_DEPARTURE"],
        errors="coerce"
    )

    departure_minutes = (
        (departure // 100) * 60
        + (departure % 100)
    )

    chunk["FLIGHT_WEATHER_TIME"] = (
        flight_date
        + pd.to_timedelta(departure_minutes, unit="m")
    ).dt.floor("h")

    # Join weather by origin airport and scheduled departure hour
    chunk = chunk.merge(
        df_weather,
        how="left",
        left_on=["ORIGIN_AIRPORT", "FLIGHT_WEATHER_TIME"],
        right_on=["airport_code", "observation_time"],
        validate="many_to_one"
    )

    # Remove temporary join columns
    chunk.drop(
        columns=["airport_code", "observation_time"],
        inplace=True,
        errors="ignore"
    )

    # Write to SQL Server
    chunk.to_sql(
        name=target_table,
        con=engine,
        schema="bronze",
        if_exists="replace" if start == 0 else "append",
        index=False,
        chunksize=1000
    )

    total_written += len(chunk)

    print(
        f"Yazıldı: {total_written:,} / "
        f"{len(df_flights_clean):,} sətir"
    )

    del chunk, flight_date, departure, departure_minutes

print("\nBirləşdirmə tamamlandı.")
print(f"Yazılmış sətirlər: {total_written:,}")
print("Cədvəl: bronze.flights_total")


Yazıldı: 80,000 / 5,810,344 sətir
Yazıldı: 160,000 / 5,810,344 sətir
Yazıldı: 240,000 / 5,810,344 sətir
Yazıldı: 320,000 / 5,810,344 sətir
Yazıldı: 400,000 / 5,810,344 sətir
Yazıldı: 480,000 / 5,810,344 sətir
Yazıldı: 560,000 / 5,810,344 sətir
Yazıldı: 640,000 / 5,810,344 sətir
Yazıldı: 720,000 / 5,810,344 sətir
Yazıldı: 800,000 / 5,810,344 sətir
Yazıldı: 880,000 / 5,810,344 sətir
Yazıldı: 960,000 / 5,810,344 sətir
Yazıldı: 1,040,000 / 5,810,344 sətir
Yazıldı: 1,120,000 / 5,810,344 sətir
Yazıldı: 1,200,000 / 5,810,344 sətir
Yazıldı: 1,280,000 / 5,810,344 sətir
Yazıldı: 1,360,000 / 5,810,344 sətir
Yazıldı: 1,440,000 / 5,810,344 sətir
Yazıldı: 1,520,000 / 5,810,344 sətir
Yazıldı: 1,600,000 / 5,810,344 sətir
Yazıldı: 1,680,000 / 5,810,344 sətir
Yazıldı: 1,760,000 / 5,810,344 sətir
Yazıldı: 1,840,000 / 5,810,344 sətir
Yazıldı: 1,920,000 / 5,810,344 sətir
Yazıldı: 2,000,000 / 5,810,344 sətir
Yazıldı: 2,080,000 / 5,810,344 sətir
Yazıldı: 2,160,000 / 5,810,344 sətir
Yazıldı: 2,240,000 / 5,810

: 

In [10]:
chunk_size = 80_000
target_table = "flights_total"

with engine.connect() as conn:
    total_written = conn.execute(
        text("SELECT COUNT(*) FROM bronze.flights_total")
    ).scalar_one()

total_rows = len(df_flights_clean)

df_weather = df_weather.drop_duplicates(
    subset=["airport_code", "observation_time"]
)

for start in range(total_written, total_rows, chunk_size):
    end = min(start + chunk_size, total_rows)
    chunk = df_flights_clean.iloc[start:end].copy()

    flight_date = pd.to_datetime({
        "year": 2015,
        "month": chunk["MONTH"],
        "day": chunk["DAY"]
    }, errors="coerce")

    departure = pd.to_numeric(
        chunk["SCHEDULED_DEPARTURE"],
        errors="coerce"
    )

    departure_minutes = (
        (departure // 100) * 60
        + (departure % 100)
    )

    chunk["FLIGHT_WEATHER_TIME"] = (
        flight_date
        + pd.to_timedelta(departure_minutes, unit="m")
    ).dt.floor("h")

    chunk = chunk.merge(
        df_weather,
        how="left",
        left_on=["ORIGIN_AIRPORT", "FLIGHT_WEATHER_TIME"],
        right_on=["airport_code", "observation_time"],
        validate="many_to_one",
        suffixes=("", "_weather")
    )

    # Weather sütunlarını əvvəlki cədvəl sxeminə uyğunlaşdır
    weather_rename = {
        "temperature": "WEATHER_TEMPERATURE",
        "precipitation": "WEATHER_PRECIPITATION",
        "wind_speed": "WEATHER_WIND_SPEED",
        "visibility": "WEATHER_VISIBILITY",
        "weather_condition": "WEATHER_CONDITION",
        "surface_pressure": "WEATHER_SURFACE_PRESSURE",
        "wind_direction": "WEATHER_WIND_DIRECTION",
        "snowfall": "WEATHER_SNOWFALL",
        "wind_gusts": "WEATHER_WIND_GUSTS"
    }

    chunk.rename(columns=weather_rename, inplace=True)

    chunk.drop(
        columns=["airport_code", "observation_time"],
        inplace=True,
        errors="ignore"
    )

    chunk.to_sql(
        name=target_table,
        con=engine,
        schema="bronze",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    total_written += len(chunk)

    print(f"Yazıldı: {total_written:,} / {total_rows:,} sətir")

    del chunk, flight_date, departure, departure_minutes
    gc.collect()

print("Birləşdirmə tamamlandı.")

: 